In [ ]:
# UPDATED METAPATH-ENHANCED ATTENTION-BASED MULTI-OMIC EXTRACTOR

import pandas as pd
import numpy as np
import os
import tensorflow.compat.v1 as tf
from collections import defaultdict
import json

# Suppress TensorFlow warnings
tf.disable_v2_behavior()
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.logging.set_verbosity(tf.logging.ERROR)

class PersistentMetapathAttentionMultiOmicExtractor:
    
    def __init__(self, hetrograph, model_path, metapath_set, mappings_dir='graph_mappings'):
        self.hetrograph = hetrograph
        self.model_path = model_path
        self.metapath_set = metapath_set
        self.mappings_dir = mappings_dir
        self.sess = None
        
        # CRITICAL: Load persistent mappings from files
        self.biological_mappings = self._load_persistent_biological_mappings()
        
        # Prepare analysis components
        self.case_labels = self._get_case_labels()
        self.nb_classes = len(self.biological_mappings['subtype'])
        
        # Prepare feature mapping and metapath analysis
        self.feature_to_omic_mapping = self._create_feature_to_omic_mapping()
        self.metapath_weights = self._initialize_metapath_weights()
        self.connectivity_matrix = self._build_connectivity_matrices()
        
        print("\n"+"="*80)
        print("Persistent Metapath-Enhanced Attention-Based Multi-Omic Extractor")
        print("="*80)
        print(f"\nModel: {model_path}")
        print(f"Mappings loaded from: {mappings_dir}")
        print(f"Cases: {len(self.case_labels)}")
        print(f"Classes: {self.nb_classes}")
        print(f"Metapaths: {len(metapath_set)} semantic pathways")
    
    def _load_persistent_biological_mappings(self):

        print("\nLoading persistent biological mappings from files...")
        
        mappings_file = os.path.join(self.mappings_dir, 'biological_mappings.json')
        
        if not os.path.exists(mappings_file):
            raise FileNotFoundError(
                f"Biological mappings file not found: {mappings_file}\n"
                f"Please run the heterograph constructor first to generate mappings!"
            )
        
        try:
            with open(mappings_file, 'r') as f:
                loaded_mappings = json.load(f)
            
            # Convert string keys back to integers for node indices
            biological_mappings = {}
            for node_type, mapping in loaded_mappings.items():
                biological_mappings[node_type] = {int(k): v for k, v in mapping.items()}
            
            print("Persistent biological mappings loaded successfully!")
            print("\nLoaded mappings:")
            for node_type, mapping in biological_mappings.items():
                print(f"  {node_type}: {len(mapping)} features")
            
            return biological_mappings
            
        except Exception as e:
            raise RuntimeError(f"\nFailed to load biological mappings: {e}")
    
    def _initialize_metapath_weights(self):
        
        metapath_weights = {}
        
        print(f"\nAnalyzing {len(self.metapath_set)} metapaths for biological significance...")
        
        for i, metapath in enumerate(self.metapath_set):
            weight = self._calculate_metapath_biological_weight(metapath)
            metapath_weights[i] = weight
            
            # Display metapath info
            metapath_str = self._metapath_to_string(metapath)
            print(f"  Metapath {i}: {metapath_str} (weight: {weight:.2f})")
        
        return metapath_weights
    
    def _calculate_metapath_biological_weight(self, metapath):
        
        weight = 1.0  # Base weight
        
        for edge in metapath:
            src, rel, dst = edge
            
            # Higher weights for direct biological connections
            if rel == 'has' and dst in ['gene', 'protein', 'mutation']:
                weight *= 1.5 
            elif rel == 'has' and dst == 'cnv':
                weight *= 1.2  
            elif rel == 'has_subtype':
                weight *= 2.0  
            elif rel == 'characterized_by':
                weight *= 1.8  
            elif rel in ['similar_to', 'coexpr_with']:
                weight *= 1.1  
            elif rel == 'in':
                weight *= 1.3  
        
        # Longer paths get slightly reduced weight (information decay)
        path_length = len(metapath)
        if path_length > 1:
            weight *= (0.95 ** (path_length - 1))
        
        return weight
    
    def _metapath_to_string(self, metapath):

        path_parts = []
        for src, rel, dst in metapath:
            if not path_parts:
                path_parts.append(src)
            path_parts.append(f"--{rel}-->{dst}")
        return " -> ".join(path_parts) if len(path_parts) > 1 else " -> ".join(path_parts)
    
    def _build_connectivity_matrices(self):

        print("Building metapath connectivity matrices...")
        
        connectivity = {}
        
        # Build connectivity for each omic type to cases
        for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
            if omic_type in self.hetrograph.node_types:
                connectivity[omic_type] = self._build_omic_case_connectivity(omic_type)
        
        return connectivity
    
    def _build_omic_case_connectivity(self, omic_type):
        
        connections = defaultdict(list)
        
        # Direct connections: case--has-->omic
        if ('case', 'has', omic_type) in self.hetrograph.edge_types:
            edge_index = self.hetrograph[('case', 'has', omic_type)].edge_index
            edge_array = self._to_numpy(edge_index)
            
            for case_idx, omic_idx in zip(edge_array[0], edge_array[1]):
                connections[int(omic_idx)].append(int(case_idx))
        
        # Reverse connections: omic--in-->case  
        if (omic_type, 'in', 'case') in self.hetrograph.edge_types:
            edge_index = self.hetrograph[(omic_type, 'in', 'case')].edge_index
            edge_array = self._to_numpy(edge_index)
            
            for omic_idx, case_idx in zip(edge_array[0], edge_array[1]):
                connections[int(omic_idx)].append(int(case_idx))
        
        print(f"  {omic_type}: {len(connections)} features connected to cases")
        return dict(connections)
    
    def _create_feature_to_omic_mapping(self):
        feature_mapping = {}
        
        # Use the PERSISTENT biological mappings to create feature mapping
        feature_idx = 0
        
        # Order MUST match the heterograph construction order
        for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
            if omic_type in self.biological_mappings:
                omic_mapping = self.biological_mappings[omic_type]
                
                # Ensure we use the indices in order
                for omic_idx in sorted(omic_mapping.keys()):
                    bio_name = omic_mapping[omic_idx]
                    feature_mapping[feature_idx] = {
                        'omic_type': omic_type,
                        'omic_index': omic_idx,
                        'biological_name': bio_name
                    }
                    feature_idx += 1
        
        print(f"\nCreated feature-to-omic mapping for {len(feature_mapping)} features using persistent mappings")
        return feature_mapping
    
    def _get_case_labels(self):
        try:
            labels = self._to_numpy(self.hetrograph['case'].y).astype(int).reshape(-1)
            return labels
        except:
            # Fallback - create dummy labels based on actual case nodes
            num_cases = self.hetrograph['case'].num_nodes
            return np.random.randint(0, self.nb_classes, size=num_cases)
    
    def extract_attention_weights(self):
        print("\n" + "="*60)
        print("Extracting Attention Weights From HAN Model")
        print("="*60)
        
        try:
            # Reset graph and create session
            tf.reset_default_graph()
            
            config = tf.ConfigProto()
            config.gpu_options.allow_growth = True
            config.allow_soft_placement = True
            
            self.sess = tf.Session(config=config)
            
            # Import the graph
            print("\nImporting graph from .meta file...")
            saver = tf.train.import_meta_graph(self.model_path + '.meta')
            
            # Restore weights
            print("Restoring weights...")
            saver.restore(self.sess, self.model_path)
            
            # Extract attention-related weights
            print("Extracting attention mechanisms...")
            attention_data = self._extract_attention_mechanisms()
            
            print("\nAttention weights extracted successfully!")
            return attention_data
            
        except Exception as e:
            print(f"\nERROR extracting attention weights: {e}")
            return None
    
    def _extract_attention_mechanisms(self):
        attention_data = {}
        
        # Get all trainable variables
        trainable_vars = tf.trainable_variables()
        print(f"\nAnalyzing {len(trainable_vars)} trainable variables for attention mechanisms...")
        
        # Find attention-related layers
        attention_layers = []
        final_layer = None
        
        for var in trainable_vars:
            try:
                var_name = var.name
                var_shape = var.shape.as_list()
                
                # Skip optimizer variables
                if any(skip in var_name.lower() for skip in ['adam', 'beta', 'global_step']):
                    continue
                
                # Extract weight values
                weight_value = self.sess.run(var)
                
                # Look for Conv1D attention layers (main attention mechanism in HAN)
                if 'conv1d' in var_name.lower() and 'kernel' in var_name.lower():
                    attention_layers.append({
                        'name': var_name,
                        'shape': var_shape,
                        'weights': weight_value,
                        'type': 'attention_conv1d'
                    })
                    
                    if len(attention_layers) <= 3:  # Log first few
                        print(f"  Found attention layer: {var_name} {var_shape}")
                
                # Look for final classification layer
                elif ('dense' in var_name.lower() and 'kernel' in var_name.lower() and 
                      len(var_shape) == 2 and var_shape[1] == self.nb_classes):
                    
                    final_layer = {
                        'name': var_name,
                        'shape': var_shape,
                        'weights': weight_value,
                        'type': 'final_classification'
                    }
                    print(f"  Found final layer: {var_name} {var_shape}")
                
            except Exception as e:
                continue
        
        attention_data['attention_layers'] = attention_layers
        attention_data['final_layer'] = final_layer
        
        print(f"  Total attention layers found: {len(attention_layers)}")
        
        return attention_data
    
    def calculate_metapath_attention_importance(self, attention_data, top_k=50):
        print("\n" + "="*60)
        print("Calculating Metapath + Attention Importance With Persistent Mappings")
        print("="*60)
        
        results = {}
        
        # Calculate for each cancer subtype using PERSISTENT mappings
        for subtype_idx in range(self.nb_classes):
            subtype_name = self.biological_mappings['subtype'][subtype_idx]
            print(f"  \nAnalyzing {subtype_name} (persistent index: {subtype_idx})...")
            
            # Get subtype-specific cases
            subtype_mask = self.case_labels == subtype_idx
            subtype_cases = np.where(subtype_mask)[0]
            
            if len(subtype_cases) == 0:
                print(f"    No cases found for {subtype_name}")
                continue
            
            print(f"    Found {len(subtype_cases)} cases for {subtype_name}")
            
            # Calculate combined importance
            feature_importance = self._calculate_combined_importance(
                attention_data, subtype_cases, subtype_name, subtype_idx, top_k
            )
            
            results[subtype_name] = feature_importance
        
        return results
    
    def _calculate_combined_importance(self, attention_data, subtype_cases, subtype_name, subtype_idx, top_k):
        
        # Initialize importance scores for each omic type
        omic_importance_scores = {
            'gene': defaultdict(lambda: {'attention': 0.0, 'metapath': 0.0, 'combined': 0.0}),
            'protein': defaultdict(lambda: {'attention': 0.0, 'metapath': 0.0, 'combined': 0.0}),
            'cnv': defaultdict(lambda: {'attention': 0.0, 'metapath': 0.0, 'combined': 0.0}),
            'mutation': defaultdict(lambda: {'attention': 0.0, 'metapath': 0.0, 'combined': 0.0})
        }
        
        # 1. Calculate attention-based importance
        print(f"    Calculating attention importance...")
        if attention_data and attention_data['attention_layers']:
            self._calculate_attention_importance(
                attention_data, omic_importance_scores, subtype_idx
            )
        
        # 2. Calculate metapath-based importance
        print(f"    Calculating metapath connectivity importance...")
        self._calculate_metapath_importance(
            subtype_cases, omic_importance_scores
        )
        
        # 3. Combine the scores
        print(f"    Combining attention + metapath scores...")
        self._combine_importance_scores(omic_importance_scores)
        
        # 4. Format results with persistent mappings
        feature_importance = self._format_combined_results(
            omic_importance_scores, subtype_name, top_k
        )
        
        return feature_importance
    
    def _calculate_attention_importance(self, attention_data, omic_importance_scores, subtype_idx):
        
        # Process each attention layer
        for layer_idx, attention_layer in enumerate(attention_data['attention_layers']):
            layer_weights = attention_layer['weights']
            layer_name = attention_layer['name']
            
            # Calculate attention coefficients for this layer
            attention_coeffs = self._extract_attention_coefficients(layer_weights, layer_name)
            
            if attention_coeffs is not None:
                # Map attention coefficients to multi-omic features
                self._map_attention_to_omic_scores(
                    attention_coeffs, omic_importance_scores, layer_idx + 1
                )
        
        # Incorporate final layer weights for subtype-specific scoring
        if attention_data['final_layer'] is not None:
            final_weights = attention_data['final_layer']['weights']
            if subtype_idx < final_weights.shape[1]:
                subtype_final_weights = final_weights[:, subtype_idx]
                self._incorporate_final_layer_attention(
                    subtype_final_weights, omic_importance_scores
                )
    
    def _calculate_metapath_importance(self, subtype_cases, omic_importance_scores):
        
        for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
            if omic_type not in self.connectivity_matrix:
                continue
            
            connectivity = self.connectivity_matrix[omic_type]
            
            for omic_idx, connected_cases in connectivity.items():
                # Calculate overlap with subtype cases
                subtype_connections = len(set(connected_cases) & set(subtype_cases))
                total_subtype_cases = len(subtype_cases)
                
                if total_subtype_cases > 0:
                    # Base connectivity score
                    connectivity_score = subtype_connections / total_subtype_cases
                    
                    # Weight by metapath importance
                    weighted_score = 0.0
                    for metapath_idx, metapath_weight in self.metapath_weights.items():
                        # Apply metapath weight to connectivity score
                        weighted_score += connectivity_score * metapath_weight
                    
                    # Normalize by number of metapaths
                    final_metapath_score = weighted_score / len(self.metapath_weights)
                    
                    omic_importance_scores[omic_type][omic_idx]['metapath'] = final_metapath_score
    
    def _combine_importance_scores(self, omic_importance_scores):
        
        # Weighting: 60% attention + 40% metapath (attention is more direct)
        attention_weight = 0.6
        metapath_weight = 0.4
        
        for omic_type in omic_importance_scores:
            for omic_idx in omic_importance_scores[omic_type]:
                scores = omic_importance_scores[omic_type][omic_idx]
                
                # Normalize scores to same scale
                attention_norm = scores['attention']
                metapath_norm = scores['metapath']
                
                # Combined score
                combined_score = (attention_weight * attention_norm + 
                                metapath_weight * metapath_norm)
                
                scores['combined'] = combined_score
    
    def _format_combined_results(self, omic_importance_scores, subtype_name, top_k):
        
        feature_importance = {'gene': [], 'protein': [], 'cnv': [], 'mutation': []}
        
        for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
            if omic_type in omic_importance_scores:
                # Get all scores for this omic type
                all_scores = [(idx, scores) for idx, scores in omic_importance_scores[omic_type].items()]
                
                if not all_scores:
                    continue
                
                # Sort by combined score
                sorted_features = sorted(all_scores, key=lambda x: x[1]['combined'], reverse=True)
                
                # Extract raw scores for normalization
                raw_combined_scores = [scores['combined'] for _, scores in sorted_features]
                raw_attention_scores = [scores['attention'] for _, scores in sorted_features]
                raw_metapath_scores = [scores['metapath'] for _, scores in sorted_features]
                
                # Apply improved scoring methods
                scaled_combined = self._apply_improved_scoring(raw_combined_scores)
                scaled_attention = self._apply_improved_scoring(raw_attention_scores)
                scaled_metapath = self._apply_improved_scoring(raw_metapath_scores)
                
                # Calculate percentiles
                percentiles = self._calculate_percentiles(raw_combined_scores)
                
                # Take top_k features with improved scores and PERSISTENT BIOLOGICAL NAMES
                for rank, ((omic_idx, scores), scaled_comb, scaled_att, scaled_meta, percentile) in enumerate(
                    zip(sorted_features[:top_k], scaled_combined[:top_k], 
                        scaled_attention[:top_k], scaled_metapath[:top_k], percentiles[:top_k])
                ):
                    # CRITICAL: Use persistent biological mappings for correct names
                    bio_name = self.biological_mappings[omic_type].get(omic_idx, f'{omic_type}_{omic_idx}')
                    
                    # Determine biological significance level
                    significance = self._get_significance_level(percentile)
                    
                    feature_importance[omic_type].append({
                        'rank': rank + 1,
                        'omic_node_index': omic_idx,
                        'omic_node_name': bio_name,  # PERSISTENT biological name
                        'omic_type': omic_type,  # Include omic type
                        'importance_score': float(scaled_comb),  # Main interpretable score (0-1000)
                        'percentile_rank': float(percentile),   # Percentile (0-100)
                        'significance_level': significance,     # High/Medium/Low
                        'attention_score': float(scaled_att),   # Scaled attention
                        'metapath_score': float(scaled_meta),   # Scaled metapath
                        'raw_combined_score': float(scores['combined']),  # Original raw score
                        'subtype': subtype_name,
                        'method': 'persistent_metapath_attention_combined'
                    })
        
        return feature_importance
    
    def _apply_improved_scoring(self, raw_scores):

        if not raw_scores or len(raw_scores) == 0:
            return raw_scores
            
        raw_scores = np.array(raw_scores)
        
        # Handle edge case where all scores are the same
        if np.std(raw_scores) == 0:
            return [1000.0] + [999.0] * (len(raw_scores) - 1)
        
        # Method 1: Scale to 0-1000 range with exponential emphasis on top features
        max_score = np.max(raw_scores)
        min_score = np.min(raw_scores)
        
        if max_score == min_score:
            return [1000.0] * len(raw_scores)
        
        # Normalize to 0-1 range first
        normalized = (raw_scores - min_score) / (max_score - min_score)
        
        # Apply power transformation to emphasize differences at the top
        # Higher power = more emphasis on top features
        power_factor = 2.0
        powered = np.power(normalized, 1.0/power_factor)
        
        # Scale to 0-1000 range
        scaled_scores = powered * 1000
        
        return scaled_scores.tolist()
    
    def _calculate_percentiles(self, raw_scores):
        """Calculate percentile ranks (0-100)"""
        if not raw_scores:
            return []
        
        # Calculate percentile rank for each score
        percentiles = []
        for score in raw_scores:
            percentile = (1.0 - (np.sum(np.array(raw_scores) > score) / len(raw_scores))) * 100
            percentiles.append(percentile)
        
        return percentiles
    
    def _get_significance_level(self, percentile):
        """Determine biological significance level based on percentile"""
        if percentile >= 95:
            return "Very High"
        elif percentile >= 80:
            return "High"
        elif percentile >= 60:
            return "Moderate"
        elif percentile >= 40:
            return "Low"
        else:
            return "Very Low"
    
    def _extract_attention_coefficients(self, layer_weights, layer_name):
        """Extract attention coefficients from layer weights"""
        
        try:
            # Handle different Conv1D weight shapes
            if len(layer_weights.shape) == 3:
                # Conv1D weights: (filter_width, input_channels, output_channels)
                if layer_weights.shape[0] == 1:
                    attention_coeffs = np.squeeze(layer_weights, axis=0)
                    attention_strength = np.linalg.norm(attention_coeffs, axis=1)
                    return attention_strength
                else:
                    attention_coeffs = np.mean(layer_weights, axis=0)
                    attention_strength = np.linalg.norm(attention_coeffs, axis=1)
                    return attention_strength
                    
            elif len(layer_weights.shape) == 2:
                attention_strength = np.linalg.norm(layer_weights, axis=1)
                return attention_strength
            
            else:
                return np.abs(layer_weights.flatten())
                
        except Exception as e:
            print(f"    Warning: Could not extract attention from {layer_name}: {e}")
            return None
    
    def _map_attention_to_omic_scores(self, attention_coeffs, omic_importance_scores, layer_weight=1.0):
        """Map attention coefficients to omic importance scores"""
        
        max_features = len(self.feature_to_omic_mapping)
        attention_coeffs = attention_coeffs[:max_features] if len(attention_coeffs) > max_features else attention_coeffs
        
        for feat_idx, attention_score in enumerate(attention_coeffs):
            if feat_idx in self.feature_to_omic_mapping:
                feature_info = self.feature_to_omic_mapping[feat_idx]
                omic_type = feature_info['omic_type']
                omic_idx = feature_info['omic_index']
                
                omic_importance_scores[omic_type][omic_idx]['attention'] += attention_score * layer_weight
    
    def _incorporate_final_layer_attention(self, final_weights, omic_importance_scores):
        """Incorporate final layer weights for subtype-specific attention weighting"""
        
        max_features = len(self.feature_to_omic_mapping)
        final_weights = final_weights[:max_features] if len(final_weights) > max_features else final_weights
        
        for feat_idx, final_weight in enumerate(final_weights):
            if feat_idx in self.feature_to_omic_mapping:
                feature_info = self.feature_to_omic_mapping[feat_idx]
                omic_type = feature_info['omic_type']
                omic_idx = feature_info['omic_index']
                
                if omic_idx in omic_importance_scores[omic_type]:
                    omic_importance_scores[omic_type][omic_idx]['attention'] *= abs(final_weight)
    
    def create_csv_output(self, feature_importance_results, output_file):
        """Create CSV file with improved interpretable scores using PERSISTENT mappings"""
        print(f"\n" + "="*60)
        print("Creating Csv Output With Persistent Biological Mappings")
        print("="*60)
        
        if not feature_importance_results:
            print("No results to save")
            return None

        output_dir = "omic_contribution_analysis"
        if not os.path.exists(output_dir):
            os.makedirs(output_dir, exist_ok=True)

        full_output_path = os.path.join(output_dir, output_file)
        
        # Prepare data for CSV
        csv_data = []
        
        for subtype_name, subtype_data in feature_importance_results.items():
            for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
                if omic_type in subtype_data:
                    for feature in subtype_data[omic_type]:
                        csv_data.append({
                            'cancer_subtype': subtype_name,
                            'omic_type': omic_type,
                            'rank': feature['rank'],
                            'feature_name': feature['omic_node_name'],  # PERSISTENT biological name
                            'feature_index': feature['omic_node_index'],
                            'importance_score': feature['importance_score'],  # 0-1000 scale
                            'percentile_rank': feature['percentile_rank'],    # 0-100 percentile
                            'significance_level': feature['significance_level'],  # Very High/High/etc
                            'attention_component': feature['attention_score'],    # Scaled attention
                            'metapath_component': feature['metapath_score'],     # Scaled metapath
                            'raw_score': feature['raw_combined_score'],         # Original raw score
                            'method': 'persistent_metapath_attention_enhanced'
                        })
        
        # Create DataFrame and save
        if csv_data:
            df = pd.DataFrame(csv_data)
            
            # Sort by subtype and importance score
            df = df.sort_values(['cancer_subtype', 'omic_type', 'importance_score'], 
                               ascending=[True, True, False])
            
            # Save to CSV
            df.to_csv(full_output_path, index=False)
            print(f" \nSaved {len(csv_data)} persistent biomarker entries to: {full_output_path}")
            
            # Display improved summary with interpretable scores
            print(f"\nImproved Summary With Persistent Biological Names:\n")
            print(f"{'Subtype':<15} {'Top Feature':<20} {'Score':<8} {'Percentile':<11} {'Significance'}")
            print("-" * 70)
            
            for subtype in df['cancer_subtype'].unique():
                subtype_df = df[df['cancer_subtype'] == subtype]
                if len(subtype_df) > 0:
                    top_feature = subtype_df.iloc[0]
                    print(f"{subtype:<15} {top_feature['feature_name']:<20} "
                          f"{top_feature['importance_score']:<8.1f} {top_feature['percentile_rank']:<11.1f} "
                          f"{top_feature['significance_level']}")
            
            # Show score distribution for interpretation
            print(f"\nScore Distribution For Interpretation:")
            all_scores = df['importance_score'].values
            print(f"  Maximum score: {np.max(all_scores):.1f}")
            print(f"  75th percentile: {np.percentile(all_scores, 75):.1f}")
            print(f"  Median score: {np.median(all_scores):.1f}")
            print(f"  25th percentile: {np.percentile(all_scores, 25):.1f}")
            print(f"  Minimum score: {np.min(all_scores):.1f}")
            
            # Count significance levels
            significance_counts = df['significance_level'].value_counts()
            print(f"\nSignificance Level Distribution:")
            for level, count in significance_counts.items():
                print(f"  {level}: {count} features")
            
            return output_file
        else:
            print("No data to save")
            return None
    
    def display_metapath_attention_results(self, feature_importance_results, top_n=5):
        """Display top metapath + attention biomarkers with PERSISTENT biological names"""
        print(f"\n" + "="*80)
        print("Top Biomarkers With Persistent Biological Mappings")
        print("="*80)

        
        for subtype_name, subtype_data in feature_importance_results.items():
            # Get the subtype index for verification
            subtype_idx = None
            for idx, name in self.biological_mappings['subtype'].items():
                if name == subtype_name:
                    subtype_idx = idx
                    break
            
            print(f"\n{subtype_name.upper()} (Persistent Index: {subtype_idx}):")
            print("-" * 70)
            
            for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
                if omic_type in subtype_data and subtype_data[omic_type]:
                    print(f"\n  Top {omic_type.upper()}S:")
                    print(f"    {'Rank':<4} {'Feature Name':<25} {'Score':<8} {'%ile':<6} {'Significance'}")
                    print(f"    {'-'*4:<4} {'-'*25:<25} {'-'*8:<8} {'-'*6:<6} {'-'*12}")
                    
                    for i, feature in enumerate(subtype_data[omic_type][:top_n]):
                        name = feature['omic_node_name']  # PERSISTENT biological name
                        importance_score = feature['importance_score']
                        percentile = feature['percentile_rank']
                        significance = feature['significance_level']
                        rank = feature['rank']
                        
                        # Truncate long names
                        display_name = name[:23] + '..' if len(name) > 25 else name
                        
                        print(f"    {rank:<4} {display_name:<25} {importance_score:<8.1f} "
                              f"{percentile:<6.1f} {significance}")
            
            # Show top overall feature for this subtype
            all_features = []
            for omic_type in ['gene', 'protein', 'cnv', 'mutation']:
                if omic_type in subtype_data:
                    all_features.extend(subtype_data[omic_type])
            
            if all_features:
                top_overall = max(all_features, key=lambda x: x['importance_score'])
                print(f"\n   TOP OVERALL: {top_overall['omic_node_name']} ({top_overall['omic_type'].upper()})")
                print(f"     Score: {top_overall['importance_score']:.1f} | "
                      f"Percentile: {top_overall['percentile_rank']:.1f} | "
                      f"Significance: {top_overall['significance_level']}")
    
    def _to_numpy(self, x):
        """Convert tensor to numpy"""
        try:
            import torch
            if isinstance(x, torch.Tensor):
                return x.cpu().numpy()
        except ImportError:
            pass
        return np.array(x)
    
    def close(self):
        """Close TensorFlow session"""
        if self.sess:
            self.sess.close()
            self.sess = None

# HELPER FUNCTION FOR METAPATH LOADING (UNCHANGED)
def load_metapath_from_csv_robust(csv_file_path, rank_number=1):
    """Load metapath set from experiment results"""
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        raise ValueError(f"CSV file not found: {csv_file_path}")
    
    if rank_number < 1 or rank_number > len(df):
        raise ValueError(f"Rank number {rank_number} is out of range. Available ranks: 1-{len(df)}")
    
    row = df.iloc[rank_number - 1]
    metapath_description = row['metapath_description']
    
    print(f"\nUsing metapath set from rank {rank_number}:")
    print(f"  Test accuracy: {row['test_accuracy']:.4f}")
    
    # Parse metapath description
    def parse_metapath_description(description):
        metapath_set = []
        path_strings = description.split(" | ")
        
        for path_string in path_strings:
            path_string = path_string.strip()
            
            if " -> " in path_string:
                metapath = []
                edges = path_string.split(" -> ")
                
                for edge in edges:
                    edge = edge.strip()
                    if "--" in edge and "-->" in edge:
                        first_dash_pos = edge.find("--")
                        last_arrow_pos = edge.rfind("-->")
                        
                        if first_dash_pos != -1 and last_arrow_pos != -1 and first_dash_pos < last_arrow_pos:
                            src = edge[:first_dash_pos]
                            rel = edge[first_dash_pos+2:last_arrow_pos]
                            dst = edge[last_arrow_pos+3:]
                            metapath.append((src, rel, dst))
                
                if metapath:
                    metapath_set.append(metapath)
            else:
                edge = path_string.strip()
                if "--" in edge and "-->" in edge:
                    first_dash_pos = edge.find("--")
                    last_arrow_pos = edge.rfind("-->")
                    
                    if first_dash_pos != -1 and last_arrow_pos != -1 and first_dash_pos < last_arrow_pos:
                        src = edge[:first_dash_pos]
                        rel = edge[first_dash_pos+2:last_arrow_pos]
                        dst = edge[last_arrow_pos+3:]
                        metapath_set.append([(src, rel, dst)])
        
        return metapath_set
    
    metapath_set = parse_metapath_description(metapath_description)
    print(f"  Parsed {len(metapath_set)} metapaths for analysis")
    
    return metapath_set

# MAIN FUNCTION
def multi_omic_contribution_analysis(hetrograph,
                                     model_path='models/final_hetegat_rank1.ckpt',
                                     metapath_csv='metapath_logs/metapath_results.csv',
                                     rank_number=1,
                                     mappings_dir='graph_mappings',
                                     output_file='multi_omic_contribution_analysis.csv',
                                     top_k=50):
   
    
    print("="*80)
    print("Extracting Biomarkers With Persistent Mappings")
    print("="*80)
    
    try:
        # Load metapath set
        metapath_set = load_metapath_from_csv_robust(metapath_csv, rank_number)
        
        if not metapath_set:
            print("\nERROR: Failed to load metapath set!")
            return None
        
        # Initialize PERSISTENT extractor
        extractor = PersistentMetapathAttentionMultiOmicExtractor(
            hetrograph, model_path, metapath_set, mappings_dir
        )
        
        # Extract attention weights from the model
        attention_data = extractor.extract_attention_weights()
        
        if not attention_data:
            print("\nFailed to extract attention weights")
            return None
        
        # Calculate combined metapath + attention importance
        feature_importance = extractor.calculate_metapath_attention_importance(attention_data, top_k)
        
        if not feature_importance:
            print("\nFailed to calculate combined importance")
            return None
        
        # Create CSV output
        output_file = extractor.create_csv_output(feature_importance, output_file)
        
        # Display results
        extractor.display_metapath_attention_results(feature_importance)
        
        print(f"\n" + "="*80)
        print("Success! Persistent Biomarkers Extracted")
        
        # Clean up
        extractor.close()
        
        return output_file
        
    except Exception as e:
        print(f"Error during persistent biomarker extraction: {e}")
        import traceback
        traceback.print_exc()
        return None